## What Is a Lift Chart?

### PART 1 — Create Sample Classification Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression


In [ ]:
# Generate dataset
X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    random_state=42
)

# Train model
model = LogisticRegression()
model.fit(X, y)

# Predicted probabilities
y_prob = model.predict_proba(X)[:, 1]

df = pd.DataFrame({
    "y_true": y,
    "y_prob": y_prob
})


### PART 2 — Create Deciles (Ranking Buckets)

In [ ]:
df = df.sort_values("y_prob", ascending=False)
df["decile"] = pd.qcut(df["y_prob"], 10, labels=False) + 1


### PART 3 — Compute Lift Table

In [ ]:
lift_table = (
    df
    .groupby("decile")
    .agg(
        total=("y_true", "count"),
        positives=("y_true", "sum")
    )
    .reset_index()
)

lift_table["response_rate"] = lift_table["positives"] / lift_table["total"]
overall_rate = df["y_true"].mean()

lift_table["lift"] = lift_table["response_rate"] / overall_rate
lift_table


### PART 4 — Basic Lift Chart (Bar Plot)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(lift_table["decile"], lift_table["lift"], color="steelblue")

plt.axhline(1, color="red", linestyle="--", label="Random Model")
plt.xlabel("Decile (1 = Highest Score)")
plt.ylabel("Lift")
plt.title("Lift Chart")
plt.legend()
plt.show()


### PART 5 — Cumulative Lift Chart (Most Common in Practice)

In [ ]:
lift_table["cum_positives"] = lift_table["positives"].cumsum()
lift_table["cum_total"] = lift_table["total"].cumsum()

lift_table["cum_response_rate"] = (
    lift_table["cum_positives"] / lift_table["cum_total"]
)

lift_table["cumulative_lift"] = (
    lift_table["cum_response_rate"] / overall_rate
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    lift_table["decile"],
    lift_table["cumulative_lift"],
    marker="o",
    label="Model"
)

plt.axhline(1, color="red", linestyle="--", label="Random")

plt.xlabel("Decile")
plt.ylabel("Cumulative Lift")
plt.title("Cumulative Lift Chart")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### PART 6 — Gains Chart (Closely Related)

In [ ]:
lift_table["cumulative_gain"] = (
    lift_table["cum_positives"] / lift_table["positives"].sum()
)

plt.figure(figsize=(8, 5))
plt.plot(
    lift_table["decile"],
    lift_table["cumulative_gain"],
    marker="o",
    label="Model"
)

plt.plot(
    lift_table["decile"],
    lift_table["decile"] / 10,
    linestyle="--",
    label="Random"
)

plt.xlabel("Decile")
plt.ylabel("Cumulative Gain")
plt.title("Gains Chart")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### PART 7 — Lift Chart Using Percentiles (More General)

In [ ]:
df["percentile"] = pd.qcut(df["y_prob"], q=10, labels=False)

lift_percentile = (
    df.groupby("percentile")
      .agg(total=("y_true", "count"),
           positives=("y_true", "sum"))
)

lift_percentile["lift"] = (
    lift_percentile["positives"] / lift_percentile["total"]
) / overall_rate


### PART 8 — Function: Create Lift Chart Easily



In [ ]:
def lift_chart(y_true, y_prob, bins=10):
    df = pd.DataFrame({"y": y_true, "prob": y_prob})
    df = df.sort_values("prob", ascending=False)
    df["bin"] = pd.qcut(df["prob"], bins, labels=False)

    summary = df.groupby("bin").agg(
        total=("y", "count"),
        positives=("y", "sum")
    )

    overall_rate = df["y"].mean()
    summary["lift"] = (summary["positives"] / summary["total"]) / overall_rate

    plt.figure(figsize=(8, 4))
    plt.bar(summary.index + 1, summary["lift"])
    plt.axhline(1, color="red", linestyle="--")
    plt.xlabel("Decile")
    plt.ylabel("Lift")
    plt.title("Lift Chart")
    plt.show()

    return summary
